# Análisis de Reclamaciones de Gastos

Este cuaderno demuestra cómo crear agentes que utilizan complementos para procesar gastos de viaje a partir de imágenes locales de recibos, generar un correo electrónico de reclamación de gastos y visualizar datos de gastos utilizando un gráfico circular. Los agentes eligen dinámicamente las funciones según el contexto de la tarea.

Pasos:
1. El Agente OCR procesa la imagen local del recibo y extrae datos de gastos de viaje.
2. El Agente de Correo genera un correo electrónico de reclamación de gastos.

### Ejemplo de un escenario de gastos de viaje:
Imagina que eres un empleado que viaja por una reunión de negocios en otra ciudad. Tu empresa tiene una política de reembolsar todos los gastos razonables relacionados con el viaje. Aquí tienes un desglose de posibles gastos de viaje:
- Transporte:
Pasajes aéreos para un viaje de ida y vuelta desde tu ciudad de origen hasta la ciudad de destino.
Taxi o servicios de transporte por aplicación hacia y desde el aeropuerto.
Transporte local en la ciudad de destino (como transporte público, alquiler de autos o taxis).

- Alojamiento:
Estancia en un hotel por tres noches en un hotel de negocios de rango medio cerca del lugar de la reunión.

- Comidas:
Asignación diaria para desayuno, almuerzo y cena, basada en la política de viáticos de la empresa.

- Gastos varios:
Tarifas de estacionamiento en el aeropuerto.
Cargos por acceso a internet en el hotel.
Propinas o pequeños cargos por servicio.

- Documentación:
Presentas todos los recibos (vuelos, taxis, hotel, comidas, etc.) y un informe de gastos completo para el reembolso.


## Importar las bibliotecas necesarias

Importa las bibliotecas y módulos necesarios para el cuaderno.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definir Modelos de Gastos

 Crea un modelo Pydantic para gastos individuales y una clase ExpenseFormatter para convertir una consulta del usuario en datos estructurados de gastos.

 Cada gasto se representará en el formato:
 `{'date': '07-Mar-2025', 'description': 'vuelo al destino', 'amount': 675.99, 'category': 'Transporte'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definiendo Herramientas - Generar el Correo Electrónico

Cree una función de herramienta para generar un correo electrónico para enviar un reclamo de gastos.  
- Esta herramienta utiliza el decorador `@tool` del Microsoft Agent Framework.  
- Calcula el monto total de los gastos y formatea los detalles en el cuerpo del correo electrónico.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Herramienta para Extraer Gastos de Viaje de Imágenes de Recibos

Cree una función de herramienta para extraer gastos de viaje de imágenes de recibos.
- Esta herramienta utiliza el decorador `@tool` del Microsoft Agent Framework.
- Lee la imagen del recibo, la codifica en base64 y devuelve el URI de datos para que el agente lo analice.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Procesamiento de Gastos

Define los agentes y conéctalos en un flujo de trabajo secuencial usando `WorkflowBuilder`.
- El agente OCR extrae datos estructurados de gastos de la imagen del recibo usando la herramienta `load_receipt_image`.
- El agente de Correo electrónico toma los datos extraídos y genera un correo profesional de reclamo de gastos usando la herramienta `generate_expense_email`.
- `WorkflowBuilder` con `add_edge` crea una cadena secuencial: Agente OCR → Agente de Correo electrónico.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Función principal

Construye el flujo de trabajo secuencial y ejecútalo para procesar la imagen del recibo y generar el correo electrónico de reclamación de gastos.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la exactitud, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional realizada por humanos. No nos hacemos responsables de cualquier malentendido o interpretación errónea derivada del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
